In [13]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
import logging
from scipy.io import savemat

In [4]:
import os
import sys

if not os.path.exists('LA-MCTS'):
    print("Cloning LA-MCTS repository...")
    !git clone https://github.com/facebookresearch/LA-MCTS.git
else:
    print("LA-MCTS repository already exists.")

%cd LA-MCTS

print("\nInstalling LA-MCTS and its dependencies...")
!pip install -q -e .

if '/content/LA-MCTS' not in sys.path:
    sys.path.append('/content/LA-MCTS')

%cd /content/

print("\n Installation complete. You can now import and use the 'lamcts' library.")


# this cell is to patch the nevergrad_sampler.py file to use np.all instead of np.alltrue (np.alltrue -> np.all)
nevergrad_path = "LA-MCTS/lamcts/sampler/nevergrad_sampler.py"
if os.path.exists(nevergrad_path):
    with open(nevergrad_path, "r") as f:
        code = f.read()
    code = code.replace("np.alltrue", "np.all")
    with open(nevergrad_path, "w") as f:
        f.write(code)
    print("Patched nevergrad_sampler.py to use np.all instead of np.alltrue.")
else:
    print("nevergrad_sampler.py not found, patch not applied.")

LA-MCTS repository already exists.
/Users/arnavoruganty/Documents/PE/LA-MCTS

Installing LA-MCTS and its dependencies...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
[Errno 2] No such file or directory: '/content/'
/Users/arnavoruganty/Documents/PE/LA-MCTS

 Installation complete. You can now import and use the 'lamcts' library.
nevergrad_sampler.py not found, patch not applied.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [5]:
import numpy as np
import torch
import random
import math
import time
import logging
import pickle
from typing import Tuple, Optional, Dict
from scipy.linalg import sqrtm, inv, dft

from lamcts import MCTS, Func, StatsFuncWrapper, ObjectFactory
from lamcts.config import SamplerEnum, ClassifierEnum, get_mcts_params, GreedyType
from lamcts.utils import set_log_level

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
def setup_ris_system(K=4, M=20, L=16, seed=42, B=None):
    """
    Sets up the RIS system parameters, channels, and fixed matrices.
    Assumes NO direct UE-BS path.
    """
    np.random.seed(seed)

    # --- System Parameters ---
    c_ue = 0.2
    c_ris = 0.4
    c_bs = 0.6
    b_min = 0.2
    alpha_ris = 2.0
    delta_ris = 0.43 * np.pi
    Pk = 1.0

    if B is None:
        B = M
    tau = K

    # --- Generate Correlation Matrices ---
    psi_bs = np.array([[c_bs ** abs(i - j) for j in range(L)] for i in range(L)], dtype=float)
    psi_ue = np.array([[c_ue ** abs(i - j) for j in range(K)] for i in range(K)], dtype=float)
    psi_irs = np.array([[c_ris ** abs(i - j) for j in range(M)] for i in range(M)], dtype=float)

    # --- Generate Base Channels ---
    H_rbar = (1 / np.sqrt(2)) * (np.random.randn(M, K) + 1j * np.random.randn(M, K))
    G_bar = (1 / np.sqrt(2)) * (np.random.randn(L, M) + 1j * np.random.randn(L, M))

    # --- Apply Spatial Correlations ---
    H_r = sqrtm(psi_irs) @ H_rbar @ sqrtm(psi_ue.T)
    G = sqrtm(psi_bs) @ G_bar @ sqrtm(psi_irs.T)

    # --- Build Equivalent Channel Correlation Matrix ---
    R_Gamma = np.kron(L * np.multiply(psi_irs, psi_irs), psi_ue)

    try:
        R_Gamma_inv = inv(R_Gamma)
    except np.linalg.LinAlgError:
        print("Warning: R_Gamma is singular or ill-conditioned. Using pseudo-inverse.")
        R_Gamma_inv = np.linalg.pinv(R_Gamma)

    # --- Generate DFT-based Orthogonal Training Sequences ---
    X = np.zeros((K, tau), dtype=complex)
    pilot_dft_base = dft(tau)

    for k_idx in range(K):
        dk_col = pilot_dft_base[:, k_idx]
        norm_sq_dk = np.linalg.norm(dk_col) ** 2
        xk_col = np.sqrt(Pk / norm_sq_dk) * dk_col
        X[k_idx, :] = xk_col.T

    # --- Non-ideal RIS Amplitude Function ---
    def amplitude_func(theta_phase):
        return ((1 - b_min) * ((np.sin(theta_phase - delta_ris) + 1) / 2) ** alpha_ris) + b_min

    return {
        'K': K,
        'M': M,
        'L': L,
        'B': B,
        'tau': tau,
        'Pk': Pk,
        'psi_bs': psi_bs,
        'psi_ue': psi_ue,
        'psi_irs': psi_irs,
        'H_r': H_r,
        'G': G,
        'R_Gamma': R_Gamma,
        'R_Gamma_inv': R_Gamma_inv,
        'X': X,
        'amplitude_func': amplitude_func,
        'b_min': b_min,
        'alpha_ris': alpha_ris,
        'delta_ris': delta_ris
    }

In [7]:
def calculate_nmse(system_params, theta_ris_phases, snr_db): 
    """ Calculates the Normalized LMMSE (NMSE) for given system parameters, RIS phases, and SNR. """ 
    from numpy.linalg import inv 
    from scipy.linalg import sqrtm 
    
    M = system_params['M'] 
    K = system_params['K'] 
    L = system_params['L'] 
    B = system_params['B'] 
    R_Gamma_inv = system_params['R_Gamma_inv'] 
    X = system_params['X'] 
    amplitude_func = system_params['amplitude_func'] 
    
    # Convert SNR to linear scale 
    snr_linear = 10**(snr_db / 10) 
    sigma2 = system_params['Pk'] / snr_linear 
    
    # Calculate V (RIS reflection coefficient matrix) 
    beta_values = amplitude_func(theta_ris_phases) 
    V = beta_values * np.exp(1j * theta_ris_phases) # (M, B) 
    
    # --- FIXED S CONSTRUCTION (valid for any B) --- 
    V_kron_IK = np.kron(np.eye(K), V) # (K*M, K*B) 
    IB_kron_X = np.kron(np.eye(B), X) # (B*K, B*tau) 
    S = V_kron_IK @ IB_kron_X # (K*M, B*tau) 
    
    # Calculate matrices for NMSE computation 
    SS_H = S @ S.conj().T # (K*M, K*M) 
    matrix_to_invert = R_Gamma_inv + (1 / (sigma2 * L)) * SS_H 
    
    try: 
        inverted_matrix = inv(matrix_to_invert) 
    except np.linalg.LinAlgError: 
        print(f"Warning: Matrix inversion failed at SNR={snr_db}dB. Using pseudo-inverse.") 
        inverted_matrix = np.linalg.pinv(matrix_to_invert) 
    
    mmse = np.trace(inverted_matrix).real 
    nmse = mmse / (L * K * M) 
    
    return nmse

In [8]:
def generate_quantized_phases(M, B, n): 
    """ Generates quantized RIS phases using 2^n discrete levels. Args: M (int): Number of RIS elements. B (int): Number of time slots/beams. n (int): Number of quantization bits (e.g., n=3 gives 8 levels). Returns: np.ndarray: (M, B) array of quantized phase values. """ 
    L = 2 ** n # Number of discrete phase levels 
    phase_levels = np.linspace(0, 2 * np.pi, num=L, endpoint=False) # Discrete phase options 
    indices = np.random.randint(0, L, size=(M, B)) # Randomly choose indices 
    return phase_levels[indices]

In [9]:
class RISOptimizeFunc(Func):
    """
    A wrapper for the RIS optimization problem that conforms to the LA-MCTS Func interface.

    This version supports an M x B dimensional search space and allows passing
    a baseline RIS phase matrix (per SNR) as the starting point.
    """

    def __init__(self, M=20, n_bits=3, snr_db=10.0, init_matrix: Optional[np.ndarray] = None):
        self._M = M
        self._n_bits = n_bits
        self._snr_db = snr_db
        self._num_levels = 2 ** self._n_bits

        self._system_params = setup_ris_system(M=self._M)
        self._B = self._system_params['B']

        self._dims = self._M * self._B
        self._lb = np.zeros(self._dims, dtype=float)
        self._ub = np.full(self._dims, self._num_levels - 1, dtype=float)

        self._quantized_phases_rad = np.linspace(
            0, 2 * np.pi * (1 - 1 / self._num_levels), self._num_levels
        )

        if init_matrix is not None:
            indices_matrix = np.argmin(
                np.abs(init_matrix[..., None] - self._quantized_phases_rad), axis=-1
            )
            self.init_x = indices_matrix.flatten()
        else:
            self.init_x = np.zeros(self._dims, dtype=int)

    @property
    def dims(self) -> int:
        return self._dims

    @property
    def lb(self) -> np.ndarray:
        return self._lb

    @property
    def ub(self) -> np.ndarray:
        return self._ub

    @property
    def is_discrete(self) -> np.ndarray:
        return np.full(self.dims, True)

    @property
    def is_minimizing(self) -> bool:
        return True

    def __call__(self, x: np.ndarray) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        batch_size = x.shape[0]
        nmse_results = np.zeros(batch_size)

        for i in range(batch_size):
            indices_matrix = x[i].astype(int).reshape((self._M, self._B))
            theta_ris_phases_matrix = self._quantized_phases_rad[indices_matrix]

            nmse_results[i] = calculate_nmse(
                self._system_params, theta_ris_phases_matrix, self._snr_db
            )

        return nmse_results, None

    def mcts_params(
        self,
        sampler: SamplerEnum = SamplerEnum.RANDOM_SAMPLER,
        classifier: ClassifierEnum = ClassifierEnum.KMEAN_SVM_CLASSIFIER
    ) -> Dict:
        params = get_mcts_params(sampler, classifier)
        params["params"]["cp"] = 0.05
        params["params"]["leaf_size"] = self.dims * 2
        params["params"]["num_init_samples"] = self.dims * 5
        params["params"]["num_samples_per_sampler"] = self.dims * 10
        return params

    def __str__(self):
        return f"RISOptimizeFunc(M={self._M}, B={self._B}, n_bits={self._n_bits})"

In [10]:
def optimize_ris_phases_sa_track(system_params, snr_db, n,
                                 base_iter=500, T_start=1.0, T_end=1e-3,
                                 alpha=None, initial_phase_matrix=None):

    M, B = system_params['M'], system_params['B']
    L = 2 ** n
    phase_levels = np.linspace(0, 2 * np.pi, L, endpoint=False)

    max_iter = base_iter * n
    if alpha is None:
        alpha = 1 - (0.01 / n)
    T_start = T_start * (1 + 0.2 * (n - 1))

    if initial_phase_matrix is not None:
        current_theta = initial_phase_matrix.copy()
    else:
        current_theta = np.random.choice(phase_levels, size=(M, B))

    current_nmse = calculate_nmse(system_params, current_theta, snr_db)
    best_theta = current_theta.copy()
    best_nmse = current_nmse
    T = T_start

    nmse_history = [current_nmse]

    for _ in range(max_iter):
        candidate_theta = current_theta.copy()

        for _ in range(np.random.randint(1, 4)):
            i = np.random.randint(0, M)
            j = np.random.randint(0, B)
            candidate_theta[i, j] = np.random.choice(phase_levels)

        candidate_nmse = calculate_nmse(system_params, candidate_theta, snr_db)
        delta = current_nmse - candidate_nmse

        if delta > 0 or np.random.rand() < np.exp(delta / T):
            current_theta = candidate_theta
            current_nmse = candidate_nmse

            if candidate_nmse < best_nmse:
                best_nmse = candidate_nmse
                best_theta = candidate_theta.copy()

        nmse_history.append(best_nmse)
        T *= alpha

        if T < T_end:
            break

    return best_theta, best_nmse, nmse_history

In [11]:
def optimize_ris_phases_pso_track(system_params, snr_db, n,
                                  num_particles=30, max_iter=200,
                                  w=0.7, c1=1.5, c2=1.5,
                                  initial_phase_matrix=None):

    M, B = system_params['M'], system_params['B']
    L = 2 ** n
    phase_levels = np.linspace(0, 2 * np.pi, L, endpoint=False)

    particles = []
    if initial_phase_matrix is not None:
        particles.append(initial_phase_matrix.copy())
        for _ in range(num_particles - 1):
            particles.append(generate_quantized_phases(M, B, n))
    else:
        particles = [generate_quantized_phases(M, B, n) for _ in range(num_particles)]

    velocities = [np.zeros((M, B)) for _ in range(num_particles)]

    pbest_positions = [p.copy() for p in particles]
    pbest_scores = [calculate_nmse(system_params, p, snr_db) for p in particles]

    gbest_index = np.argmin(pbest_scores)
    gbest_position = pbest_positions[gbest_index].copy()
    gbest_score = pbest_scores[gbest_index]

    nmse_history = [gbest_score]

    for _ in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(), np.random.rand()

            velocities[i] = (w * velocities[i] +
                             c1 * r1 * (pbest_positions[i] - particles[i]) +
                             c2 * r2 * (gbest_position - particles[i]))

            particles[i] += velocities[i]

            particles[i] = phase_levels[
                np.argmin(np.abs(phase_levels[:, None, None] - particles[i]), axis=0)
            ]

            score = calculate_nmse(system_params, particles[i], snr_db)

            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = particles[i].copy()

                if score < gbest_score:
                    gbest_score = score
                    gbest_position = particles[i].copy()

        nmse_history.append(gbest_score)

    return gbest_position, gbest_score, nmse_history


In [15]:
import os

print("Working directory:", os.getcwd())
print("Directory contents before:", os.listdir(os.getcwd()))

OUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("Saving outputs to:", OUT_DIR)

Working directory: /Users/arnavoruganty/Documents/PE/LA-MCTS
Directory contents before: ['CODE_OF_CONDUCT.md', 'NMSE_convergence_M20_n2_SNR10dB.eps', 'ris_optimization_results_snr_10_nevergrad_sampler.pkl', '.DS_Store', 'LICENSE', 'CHANGELOG.md', 'example', 'docs', 'README.md', 'setup.py', '.gitignore', 'lamcts', 'CONTRIBUTING.md', 'NMSE_convergence_M20_n2_SNR10dB.mat', 'LA_MCTS.egg-info', 'NMSE_convergence_M20_n2_SNR10dB.pdf', '.git', 'ris_optimization_results']
Saving outputs to: /Users/arnavoruganty/Documents/PE/LA-MCTS/outputs


In [19]:
# ======================================================
# System Configuration
# ======================================================
M = 20
N_BITS = 2
snr_db = 10
CALL_BUDGET = 5000
PSO_NUM_PARTICLES = 30

sampler = SamplerEnum.NEVERGRAD_SAMPLER
CLASSIFIER = ClassifierEnum.KMEAN_SVM_CLASSIFIER
OUTPUT_FILE_PREFIX = "ris_optimization_results"

logging.basicConfig(level=logging.INFO)

# ======================================================
# LA-MCTS OPTIMIZATION
# ======================================================
all_results = {}

baseline_matrix = generate_quantized_phases(M, M, N_BITS)

sampler_name = sampler.name.lower()
print(f"\n--- Running LA-MCTS optimization for SNR = {snr_db} dB ---")

ris_func = RISOptimizeFunc(
    M=M,
    n_bits=N_BITS,
    snr_db=snr_db,
    init_matrix=baseline_matrix
)

func_wrapper = StatsFuncWrapper(ris_func)
mcts_params = ris_func.mcts_params(
    sampler=sampler,
    classifier=CLASSIFIER
)

mcts = MCTS.create_mcts(func_wrapper, func_wrapper, mcts_params)

# Baseline NMSE (1st evaluation)
fval, _ = ris_func(ris_func.init_x.reshape(1, -1))
baseline_nmse = fval[0]

func_wrapper.baseline_nmse = baseline_nmse
func_wrapper.baseline_x = ris_func.init_x

print(f"Baseline NMSE: {baseline_nmse:.6f}")

start_time = time.time()
try:
    stats = mcts.search(
        greedy=GreedyType.Best,
        call_budget=CALL_BUDGET
    )
except Exception as e:
    print(f"Error during LA-MCTS search: {e}")
    import traceback
    traceback.print_exc()
    stats = func_wrapper.stats

end_time = time.time()
print(f"Search finished in {end_time - start_time:.2f} seconds.")

if stats and len(stats.call_history) > 0:
    output_file = f"{OUTPUT_FILE_PREFIX}_snr_{snr_db}_{sampler_name}.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(stats, f)
else:
    print("No valid LA-MCTS results found.")

# ======================================================
# SA AND PSO OPTIMIZATION
# ======================================================
print("\n--- Running SA and PSO optimizations ---")

system_params = setup_ris_system(
    K=4, M=M, L=16, seed=42, B=M
)

initial_matrix = baseline_matrix

theta_sa, best_nmse_sa, sa_history = optimize_ris_phases_sa_track(
    system_params,
    snr_db,
    N_BITS,
    initial_phase_matrix=initial_matrix,
    base_iter=1000 // N_BITS
)

theta_pso, best_nmse_pso, pso_history = optimize_ris_phases_pso_track(
    system_params,
    snr_db,
    N_BITS,
    initial_phase_matrix=initial_matrix,
    num_particles=PSO_NUM_PARTICLES,
    max_iter=200
)

print(f"SA Best NMSE:  {best_nmse_sa:.6f}")
print(f"PSO Best NMSE: {best_nmse_pso:.6f}")

# ======================================================
# CONVERGENCE PLOT (IEEE STYLE)
# ======================================================
plt.rcParams.update({
    "font.size": 12,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.5,
})

fig, ax = plt.subplots(figsize=(8, 5))

# --------------------------------------
# Determine max evaluations
# --------------------------------------
max_evals = CALL_BUDGET

# SA evals (1 eval per iteration)
sa_evals = np.arange(1, len(sa_history) + 1)
max_evals = max(max_evals, sa_evals[-1])

# PSO evals (num_particles evals per iteration)
pso_evals = np.arange(1, len(pso_history) + 1) * PSO_NUM_PARTICLES
max_evals = max(max_evals, pso_evals[-1])

# --------------------------------------
# Helper: extend curves to max_evals
# --------------------------------------
def extend_curve(x, y, max_x):
    if x[-1] < max_x:
        x = np.append(x, max_x)
        y = np.append(y, y[-1])
    return x, y

# --------------------------------------
# LA-MCTS curve
# --------------------------------------
try:
    output_file = f"{OUTPUT_FILE_PREFIX}_snr_{snr_db}_nevergrad_sampler.pkl"
    with open(output_file, "rb") as f:
        loaded_stats = pickle.load(f)

    call_marks = np.array([1] + [cp.call_mark for cp in loaded_stats.call_history])
    best_fx = np.array([baseline_nmse] + [cp.fx for cp in loaded_stats.call_history])

    call_marks, best_fx = extend_curve(call_marks, best_fx, max_evals)

    ax.plot(call_marks, best_fx, linestyle="--", linewidth=3.0, label="LA-MCTS-PO")
except Exception as e:
    print(f"Skipping LATS plot: {e}")
    call_marks = np.array([])
    best_fx = np.array([])

# --------------------------------------
# SA curve
# --------------------------------------
sa_evals, sa_history_plot = extend_curve(sa_evals, np.array(sa_history), max_evals)
ax.plot(sa_evals, sa_history_plot, linestyle="--", linewidth=3.0, label="SA")

# --------------------------------------
# PSO curve
# --------------------------------------
pso_evals, pso_history_plot = extend_curve(pso_evals, np.array(pso_history), max_evals)
ax.plot(pso_evals, pso_history_plot, linestyle="--", linewidth=3.0, label="PSO")

# --------------------------------------
# Formatting
# --------------------------------------
ax.set_xlabel("Function Evaluations", fontweight="bold")
ax.set_ylabel("NMSE", fontweight="bold")
ax.set_yscale("log")
ax.grid(True, which="both", linestyle="--", linewidth=0.8)

legend = ax.legend(fontsize=10, frameon=True)
for text in legend.get_texts():
    text.set_fontweight("bold")

ax.set_xlim(0, max_evals)

# No title (IEEE style)
fig.tight_layout()

fig.savefig(f"NMSE_convergence_M{M}_n{N_BITS}_SNR{snr_db}dB.pdf", dpi=300)
fig.savefig(f"NMSE_convergence_M{M}_n{N_BITS}_SNR{snr_db}dB.eps", format="eps")

plt.close(fig)

# --------------------------------------
# Export .mat for MATLAB editing
# --------------------------------------
savemat(
    f"NMSE_convergence_M{M}_n{N_BITS}_SNR{snr_db}dB.mat",
    {
        "call_marks": call_marks,
        "best_fx": best_fx,
        "sa_evals": sa_evals,
        "sa_history": sa_history_plot,
        "pso_evals": pso_evals,
        "pso_history": pso_history_plot,
        "max_evals": max_evals,
        "baseline_nmse": baseline_nmse
    }
)


--- Running LA-MCTS optimization for SNR = 10 dB ---
Baseline NMSE: 0.103073
Search finished in 175.03 seconds.

--- Running SA and PSO optimizations ---


2025-12-26 17:46:17,173 - matplotlib.backends.backend_ps - WARNING - The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


SA Best NMSE:  0.078294
PSO Best NMSE: 0.061641
